# Customer Propensity Models

Trains two models using **Feature Store** features via `generate_training_set()`:
1. **Repurchase Propensity** (XGBClassifier) — 90-day repurchase prediction
2. **Customer LTV** (XGBRegressor) — 12-month spend prediction

Training submitted as **ML Job** on compute pool.
Models logged to **Model Registry**, scores to `SCORING.CUSTOMER_SCORES`.


In [ ]:
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.registry import Registry

session = Session.builder.configs({"connection_name": "my_connection"}).create()
session.use_database('SB_COMMAND_CENTER')
session.use_warehouse('COMPUTE_WH')

fs = FeatureStore(
    session=session,
    database='SB_COMMAND_CENTER',
    name='FEATURE_STORE',
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST
)

reg = Registry(session, database_name='SB_COMMAND_CENTER', schema_name='REGISTRY')
print("Session + Feature Store + Registry ready")

## Build Training Labels

Create labels from raw transaction history:
- **Repurchase**: did customer transact within 90 days after a reference cutoff?
- **LTV**: total spend in the 12 months after the cutoff


In [ ]:
# Use a reference date relative to the DATA, not today's date
# Data spans Jan 2024 - Jun 2025. Set cutoff ~9 months before end of data.
date_range = session.sql("""
    SELECT MIN(TRANSACTION_DATE) AS earliest, MAX(TRANSACTION_DATE) AS latest
    FROM SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
""").collect()[0]
print(f"Data range: {date_range['EARLIEST']} to {date_range['LATEST']}")

labels_df = session.sql("""
WITH data_bounds AS (
    SELECT MAX(TRANSACTION_DATE) AS max_date
    FROM SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
),
customer_first_seen AS (
    SELECT CUSTOMER_ID, MIN(TRANSACTION_DATE) AS first_seen
    FROM SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
    GROUP BY CUSTOMER_ID
),
reference AS (
    -- Cutoff = 6 months before end of data (leaves room for 90-day + LTV window)
    SELECT c.CUSTOMER_ID,
           DATEADD('month', -6, d.max_date)::TIMESTAMP_NTZ AS reference_date
    FROM customer_first_seen c
    CROSS JOIN data_bounds d
    WHERE c.first_seen < DATEADD('month', -6, d.max_date)
),
repurchase_label AS (
    SELECT r.CUSTOMER_ID, r.reference_date,
           MAX(CASE WHEN t.TRANSACTION_DATE BETWEEN r.reference_date
                    AND DATEADD('day', 90, r.reference_date) THEN 1 ELSE 0 END) AS REPURCHASED_90D
    FROM reference r
    LEFT JOIN SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS t
        ON r.CUSTOMER_ID = t.CUSTOMER_ID
    GROUP BY r.CUSTOMER_ID, r.reference_date
),
ltv_label AS (
    SELECT r.CUSTOMER_ID,
           COALESCE(SUM(CASE WHEN t.TRANSACTION_DATE BETWEEN r.reference_date
                             AND DATEADD('month', 6, r.reference_date)
                        THEN t.PURCHASE_AMOUNT ELSE 0 END), 0) AS LTV_6M
    FROM reference r
    LEFT JOIN SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS t
        ON r.CUSTOMER_ID = t.CUSTOMER_ID
    GROUP BY r.CUSTOMER_ID
)
SELECT rp.CUSTOMER_ID, rp.REPURCHASED_90D, lt.LTV_6M AS LTV_12M
FROM repurchase_label rp
JOIN ltv_label lt ON rp.CUSTOMER_ID = lt.CUSTOMER_ID
""")

print(f'Label rows: {labels_df.count()}')
labels_df.select(
    F.avg('REPURCHASED_90D').alias('repurchase_rate'),
    F.avg('LTV_12M').alias('avg_ltv')
).show()

## Retrieve Features from Feature Store

Use `generate_training_set()` to join customer features from both
`CUSTOMER_RFM_FV` and `CUSTOMER_BEHAVIOR_FV` onto the label spine.
No `spine_timestamp_col` since these feature views are not time-series.


In [ ]:
rfm_fv = fs.get_feature_view('CUSTOMER_RFM_FV', 'v1')
behavior_fv = fs.get_feature_view('CUSTOMER_BEHAVIOR_FV', 'v1')

print(f"RFM features: {rfm_fv.feature_names}")
print(f"Behavior features: {behavior_fv.feature_names}")

# Spine = customers with their labels
spine_df = labels_df.select('CUSTOMER_ID')

# generate_training_set joins features onto the spine by CUSTOMER_ID
training_data = fs.generate_training_set(
    spine_df=spine_df,
    features=[rfm_fv, behavior_fv],
    spine_label_cols=[]
)

# Join labels back
training_full = training_data.join(labels_df, on='CUSTOMER_ID')
print(f'Training set: {training_full.count()} rows')

# Materialize for ML Job
training_full.write.mode('overwrite').save_as_table(
    'SB_COMMAND_CENTER.SCORING.CUSTOMER_TRAINING_DATA'
)
print("Materialized to SCORING.CUSTOMER_TRAINING_DATA")

## Submit Training as ML Job

Trains both models on a compute pool via `@remote`.


In [ ]:
from snowflake.ml.jobs import remote

@remote(
    compute_pool='ML_CPU_POOL',
    stage_name='SB_COMMAND_CENTER.FORECASTING.ML_STAGE',
    pip_requirements=['xgboost', 'scikit-learn', 'pandas', 'numpy'],
    session=session,
)
def train_customer_models():
    import pandas as pd
    import numpy as np
    from xgboost import XGBClassifier, XGBRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, mean_absolute_error
    from datetime import datetime
    from snowflake.snowpark import Session as SpSession
    from snowflake.ml.registry import Registry

    sp = SpSession.builder.getOrCreate()

    pdf = sp.table('SB_COMMAND_CENTER.SCORING.CUSTOMER_TRAINING_DATA').to_pandas()

    FEATURE_COLS = [
        'RECENCY_DAYS', 'ORDER_FREQUENCY', 'TOTAL_MONETARY', 'AVG_ORDER_VALUE',
        'BRAND_DIVERSITY', 'DISCOUNT_SENSITIVITY', 'PROREWARDS_FLAG',
        'AVG_BASKET_SIZE', 'AVG_BASKET_ITEMS', 'COUPON_USAGE_RATE'
    ]
    pdf[FEATURE_COLS] = pdf[FEATURE_COLS].fillna(0)

    X = pdf[FEATURE_COLS]
    y_rep = pdf['REPURCHASED_90D']
    y_ltv = pdf['LTV_12M']

    X_train, X_test, y_train_rep, y_test_rep = train_test_split(
        X, y_rep, test_size=0.2, random_state=42, stratify=y_rep
    )
    _, _, y_train_ltv, y_test_ltv = train_test_split(
        X, y_ltv, test_size=0.2, random_state=42
    )

    # Repurchase classifier
    rep_model = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        scale_pos_weight=(y_train_rep == 0).sum() / max((y_train_rep == 1).sum(), 1),
        eval_metric='auc', random_state=42
    )
    rep_model.fit(X_train, y_train_rep, eval_set=[(X_test, y_test_rep)], verbose=False)
    rep_auc = roc_auc_score(y_test_rep, rep_model.predict_proba(X_test)[:, 1])

    # LTV regressor
    ltv_model = XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        eval_metric='mae', random_state=42
    )
    ltv_model.fit(X_train, y_train_ltv, eval_set=[(X_test, y_test_ltv)], verbose=False)
    ltv_mae = mean_absolute_error(y_test_ltv, ltv_model.predict(X_test))

    # Log both to registry
    reg = Registry(sp, database_name='SB_COMMAND_CENTER', schema_name='REGISTRY')
    sample = pd.DataFrame([X_train.iloc[0].to_dict()])

    reg.log_model(
        rep_model,
        model_name='customer_repurchase_xgb',
        version_name=f"v{datetime.now().strftime('%Y%m%d_%H%M')}",
        metrics={'auc': float(rep_auc), 'train_rows': len(X_train), 'test_rows': len(X_test)},
        sample_input_data=sample,
    )
    reg.log_model(
        ltv_model,
        model_name='customer_ltv_xgb',
        version_name=f"v{datetime.now().strftime('%Y%m%d_%H%M')}",
        metrics={'mae': float(ltv_mae), 'train_rows': len(X_train), 'test_rows': len(X_test)},
        sample_input_data=sample,
    )

    # Score full customer base
    all_pdf = sp.table('SB_COMMAND_CENTER.SCORING.CUSTOMER_TRAINING_DATA').to_pandas()
    all_pdf[FEATURE_COLS] = all_pdf[FEATURE_COLS].fillna(0)
    all_pdf['REPURCHASE_SCORE'] = rep_model.predict_proba(all_pdf[FEATURE_COLS])[:, 1]
    all_pdf['LTV_PREDICTION'] = np.clip(ltv_model.predict(all_pdf[FEATURE_COLS]), 0, None)
    all_pdf['SEGMENT_LABEL'] = pd.cut(
        all_pdf['REPURCHASE_SCORE'],
        bins=[0, 0.3, 0.6, 0.8, 1.0],
        labels=['At Risk', 'Moderate', 'Engaged', 'Champion']
    )

    output = all_pdf[['CUSTOMER_ID', 'REPURCHASE_SCORE', 'LTV_PREDICTION', 'SEGMENT_LABEL']].copy()
    output['SCORED_AT'] = datetime.now()

    scores_sdf = sp.create_dataframe(output)
    scores_sdf.write.mode('overwrite').save_as_table('SB_COMMAND_CENTER.SCORING.CUSTOMER_SCORES')

    return {
        'repurchase_auc': float(rep_auc),
        'ltv_mae': float(ltv_mae),
        'customers_scored': len(output),
        'segments': output['SEGMENT_LABEL'].value_counts().to_dict(),
    }

print("Training function defined — will run on ML_CPU_POOL")

In [ ]:
# Submit and wait
job = train_customer_models()
result = job.result()

print(f"Repurchase AUC: {result['repurchase_auc']:.4f}")
print(f"LTV MAE: ${result['ltv_mae']:.2f}")
print(f"Customers scored: {result['customers_scored']}")
print(f"Segments: {result['segments']}")

## Verify


In [ ]:
session.table('SB_COMMAND_CENTER.SCORING.CUSTOMER_SCORES').group_by(
    'SEGMENT_LABEL'
).agg(
    F.count('*').alias('CUSTOMERS'),
    F.round(F.avg('REPURCHASE_SCORE'), 3).alias('AVG_REPURCHASE_SCORE'),
    F.round(F.avg('LTV_PREDICTION'), 2).alias('AVG_LTV')
).sort(F.col('AVG_REPURCHASE_SCORE').desc()).show()

In [ ]:
# Check registered models
session.sql("SHOW MODELS IN SCHEMA SB_COMMAND_CENTER.REGISTRY").show()